<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 04. Algoritmo k-NN — "Dime con quién andas y te diré quién eres"
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 08
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/08%20-%20Classification/Para%20Dummies/04_KNN_Clasificacion_y_Seleccion_Modelos_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

El **algoritmo k-NN** es el más intuitivo de todos los clasificadores — no necesita ecuaciones para entenderlo. Aprenderás:

1. Cómo funciona k-NN con la analogía del barrio.
2. Por qué el escalado es **obligatorio** para k-NN.
3. Cómo elegir el mejor valor de `k` con validación cruzada.
4. Qué es un **Pipeline** (proceso automatizado de limpieza + modelo).

---
## 1. La analogía del barrio 🏘️

El refrán lo resume perfecto: **"Dime con quién andas y te diré quién eres"**.

El algoritmo k-NN funciona exactamente así:

1. Llega un paciente nuevo con sus datos (edad, presión, colesterol...).
2. El algoritmo busca los **k pacientes más parecidos** en la base de datos histórica.
3. Mira qué diagnóstico tienen esos k vecinos más cercanos.
4. El paciente nuevo recibe el diagnóstico **más votado** entre sus vecinos.

```
       Paciente NUEVO ★
       (edad=55, presión=140)
              │
   Busca los 3 vecinos más cercanos (k=3)
              │
        ┌─────┴─────┐
       🔴          🔴          🟢
    Cardiopatía  Cardiopatía  Sano
              │
   Votación: 2 Cardiopatía vs 1 Sano
              │
   Predicción: ★ → 🔴 Cardiopatía
```

> 💡 **k = ¿cuántos vecinos consultar?** Si k=1, solo miras tu vecino más cercano. Si k=50, consultas a muchos — más estable pero menos preciso para casos raros.

In [ ]:
import os, urllib.parse, urllib.request, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

def load_dataset(filename, module_name="08 - Classification"):
    candidates = [f"data/{filename}", f"../{module_name}/data/{filename}", f"{module_name}/data/{filename}", filename]
    for path in candidates:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    target_path = f"data/{filename}"
    url = f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/{urllib.parse.quote(module_name)}/data/{urllib.parse.quote(filename)}"
    urllib.request.urlretrieve(url, target_path)
    return target_path

iris = pd.read_csv(load_dataset('iris.csv'))
le = LabelEncoder()
X = iris.drop(columns=['species'])
y = le.fit_transform(iris['species'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"✅ Dataset Iris listo: {len(X_train)} train · {len(X_test)} test")

---
## 2. Por qué el escalado es OBLIGATORIO para k-NN ⚠️

k-NN mide **distancias**. Si una variable va de 0 a 1 y otra va de 0 a 1000, la segunda dominará completamente el cálculo de distancias — aunque no sea la más importante.

Es como medir la distancia entre dos ciudades combinando kilómetros y milímetros en la misma suma: los milímetros serían invisibles.

In [ ]:
# k-NN SIN escalar
knn_crudo = KNeighborsClassifier(n_neighbors=5)
knn_crudo.fit(X_train, y_train)
acc_crudo = accuracy_score(y_test, knn_crudo.predict(X_test))

# k-NN CON escalado (correcto)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

knn_escalado = KNeighborsClassifier(n_neighbors=5)
knn_escalado.fit(X_train_sc, y_train)
acc_escalado = accuracy_score(y_test, knn_escalado.predict(X_test_sc))

print("📏 Impacto del escalado en k-NN (k=5):")
print(f"   Sin StandardScaler: {acc_crudo:.1%}")
print(f"   Con StandardScaler: {acc_escalado:.1%}  ← siempre mejor (o igual)")
print(f"\n⚠️  REGLA: k-NN SIEMPRE necesita escalado previo.")
print("   Aplica fit_transform en train y solo transform en test.")

---
## 3. Encontrar el mejor k — La curva de sesgo-varianza 📈

In [ ]:
rango_k = range(1, 31)
acc_train_list, acc_test_list = [], []

for k in rango_k:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_sc, y_train)
    acc_train_list.append(accuracy_score(y_train, knn.predict(X_train_sc)))
    acc_test_list.append(accuracy_score(y_test, knn.predict(X_test_sc)))

mejor_k = rango_k[np.argmax(acc_test_list)]

plt.figure(figsize=(11, 5))
plt.plot(rango_k, acc_train_list, 'o-', color='#6366f1', linewidth=2, label='Entrenamiento', markersize=5)
plt.plot(rango_k, acc_test_list,  's-', color='#f59e0b', linewidth=2.5, label='Prueba', markersize=6)
plt.axvline(mejor_k, color='#ef4444', linestyle='--', linewidth=2, label=f'Mejor k = {mejor_k}')
plt.xlabel('Valor de k (número de vecinos)', fontsize=12)
plt.ylabel('Exactitud (Accuracy)', fontsize=12)
plt.title('🔍 Impacto de k en el rendimiento del modelo\n(k pequeño = sobreajuste, k grande = subajuste)',
          fontweight='bold', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 k=1: el modelo memoriza todo (sobreajuste) — perfecto en train, peor en test")
print(f"   k grande: el modelo es muy general (subajuste) — peor en ambos")
print(f"   ✅ Mejor k encontrado: k = {mejor_k} (Acc. test: {max(acc_test_list):.1%})")

---
## 4. Pipeline — El proceso automatizado en 2 pasos 🏭

Un **Pipeline** es como una cadena de producción automatizada:
1. Entrada: datos crudos.
2. Paso 1: Escalado automático.
3. Paso 2: Clasificación con k-NN.
4. Salida: predicción.

La ventaja: ¡nunca te olvidas de escalar los datos de prueba! Todo sucede automáticamente.

In [ ]:
# Pipeline: escalado + k-NN en un solo objeto
pipeline = Pipeline([
    ('escalado', StandardScaler()),
    ('knn',      KNeighborsClassifier())
])

# Búsqueda del mejor k con validación cruzada estratificada
param_grid = {'knn__n_neighbors': list(range(1, 21))}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
busqueda = GridSearchCV(pipeline, param_grid, cv=cv, scoring='accuracy')
busqueda.fit(X_train, y_train)  # le pasamos los datos SIN escalar — el Pipeline lo hace solo

mejor_k_cv = busqueda.best_params_['knn__n_neighbors']
mejor_acc  = busqueda.best_score_

print(f"🏆 GridSearchCV encontró el mejor k = {mejor_k_cv}")
print(f"   Accuracy en validación cruzada: {mejor_acc:.1%}")
print(f"   Accuracy en conjunto de prueba: {accuracy_score(y_test, busqueda.predict(X_test)):.1%}")

print(f"\n✅ Ventajas del Pipeline:")
print("   1. Nunca te olvidas de escalar los datos de prueba.")
print("   2. El GridSearchCV ajusta los hiperparámetros automáticamente.")
print("   3. Todo el proceso (escalar + predecir) queda en un solo objeto.")

---
## 5. Resumen final del módulo 08 — Clasificación 🎓

| Algoritmo | Cómo funciona | Necesita escalado | Cuándo usarlo |
|---|---|:---:|---|
| **Regresión Logística** | Función Sigmoide + coeficientes lineales | Sí | Línea base, interpretabilidad |
| **Logística Multiclase** | OvR o Softmax | Sí | 3+ clases |
| **k-NN** | Votación de vecinos más cercanos | ✅ **Sí, obligatorio** | Datos no lineales, pocos ejemplos |

**Pipeline de clasificación óptimo:**
```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  KNeighborsClassifier(n_neighbors=k_optimo))
])
pipeline.fit(X_train, y_train)
pipeline.predict(X_test)  # escala y predice automáticamente
```

> 🏁 **¡Felicidades!** Completaste el módulo 08 de Clasificación. Ahora sabes predecir categorías, evaluar modelos correctamente y elegir el mejor k para k-NN.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos</i>
  </p>
</div>